# Scaling up your analysis - multi-threading and distributed RDataFrame

## Multi-threaded RDataFrame

RDataFrame can run the event loop using several CPU cores.

To enable this, call:

```python
ROOT.EnableImplicitMT()
```

before constructing the RDataFrame.

RDataFrame will split the event range into chunks, process them in parallel, and merge the partial results.

This is often an easy way to speed up an analysis, especially when reading local files and filling histograms.

### Important caveat

Multi-threading is safe for standard RDataFrame operations such as:

- `Histo1D`
- `Count`
- `Mean`
- `Snapshot`

But user-defined code must be thread-safe.

Avoid modifying global Python or C++ state inside `Define`, `Filter`, or custom functions. A good rule is:

> A helper function should take inputs and return an output, without changing anything outside the function.

In [ ]:
import ROOT

In [ ]:
%%time

# Baseline: single-thread execution
treename = "Events"
filename = "data/particles.root"

df = ROOT.RDataFrame(treename, filename)
df_selected = df.Define("good_pt", "sqrt(px*px + py*py)[E > 100]")

mean_good_pt = df_selected.Mean("good_pt")
mean_good_pt.GetValue()

In [ ]:
%%time

# Enable implicit multi-threading.
# By default ROOT chooses the number of worker threads.
ROOT.EnableImplicitMT()

treename = "Events"
filename = "data/particles.root"

df = ROOT.RDataFrame(treename, filename)
df_selected = df.Define("good_pt", "sqrt(px*px + py*py)[E > 100]")

mean_good_pt = df_selected.Mean("good_pt")
mean_good_pt.GetValue()

# Disable it again so the rest of the notebook behaves predictably.
ROOT.DisableImplicitMT()

The speedup depends on:
- dataset size,
- storage system,
- decompression cost,
- number of cores,
- the actual analysis operations.

## Distributed RDataFrame

An `RDataFrame` analysis written in Python can be executed both *locally* - possibly in parallel on the cores of the machine - and *distributedly* by offloading computations to external resources, which include:

- [Spark](https://spark.apache.org/) and 
- [Dask](https://dask.org/) clusters. 

- This feature is enabled by the architecture depicted below.

- It shows that RDataFrame computation graphs can be mapped to different kinds of resources via backends.

- In this notebook we will exercise the Dask backend, which divides an `RDataFrame` input dataset in logical ranges and submits computations for each of those ranges to Dask resources.

<img src="data/DistRDF_architecture.png" alt="Distributed RDataFrame">

## Create a Dask client (in a dummy `LocalCluster` created inside the notebook)

- In order to work with a Dask cluster we need a `Client` object.
- It represents the connection to that cluster and allows to configure execution-related parameters (e.g. number of cores, memory). 
- The client object is just the intermediary between our client session and the cluster resources.
- Dask supports many different resource managers.
- We will follow the [Dask documentation](https://distributed.dask.org/en/stable/client.html) regarding the creation of a `Client`.

In [ ]:
from distributed import Client, LocalCluster
# A LocalCluster creates a test Cluster, segmenting the resources available under your notebook
# It is meant for prototyping purposes and will not give full performance
cluster = LocalCluster(n_workers=2, threads_per_worker=1, processes=True, memory_limit="2GiB")
client = Client(cluster)

## Create a ROOT dataframe

We now create a distributed RDataFrame with Dask. It accepts two more keyword arguments:
- the number of partitions to apply to the dataset (`npartitions`).
- the `Client` object (`daskclient`).

Besides these details, a Dask RDataFrame is not different from a local RDataFrame: the analysis presented in this notebook would not change if we wanted to execute it locally.

In [ ]:
import ROOT

df = ROOT.RDataFrame("Events",
                     "data/particles.root",
                     npartitions=4,
                     executor=client)

## Run your analysis unchanged

- From now on, the rest of your application can be written **exactly** as we have seen with local RDataFrame. 

- The goal of the distributed RDataFrame module is to support all the traditional RDataFrame operations (those that make sense in a distributed context at least). 

- Currently only a subset of those is available and can be found in the corresponding [section of the documentation](https://root.cern/doc/master/classROOT_1_1RDataFrame.html#distrdf)

In [ ]:
%%time
df1 = df.Filter("nPart > 0")
df2 = df.Define("good_pt", "sqrt(px*px + py*py)[E > 100]")
count = df.Count()
mean_good_pt = df2.Mean("good_pt")
mean_px = df2.Mean("px")
mean_py = df2.Mean("py")
print(f"Number of events after processing: {count.GetValue()}")
print(f"Mean of column 'good_pt': {mean_good_pt.GetValue()}")
print(f"Mean of column 'px': {mean_px.GetValue()}")
print(f"Mean of column 'py': {mean_py.GetValue()}")


Example code for `HTCondorCluster`(not executable here, as we do not have access to an HTCondor cluster).

It can work on `lxplus`.

In [ ]:
from dask.distributed import LocalCluster, Client
from dask_jobqueue import HTCondorCluster

cluster = HTCondorCluster(
    cores=1,
    memory='2000MB',
    disk='1000MB',
)

# Use the scale method to send as many jobs as needed
cluster.scale(4)
client = Client(cluster)

df = ROOT.RDataFrame("Events",
                     "data/particles.root",
                     npartitions=4,
                     executor=client)

